In [1]:
!pip install plotly pandas numpy

In [21]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

All libraries imported successfully!


In [5]:
from google.colab import files
uploaded = files.upload()
df = pd.read_csv('netflix_titles.csv')

Saving netflix_titles.csv to netflix_titles (1).csv


In [22]:
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"\nTotal Records : {df.shape[0]}")
print(f"Total Columns : {df.shape[1]}")
print(f"\nColumn Names:\n{list(df.columns)}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nMissing Values:\n{df.isnull().sum()}")

DATASET OVERVIEW

Total Records : 8797
Total Columns : 16

Column Names:
['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description', 'year_added', 'month_added', 'duration_int', 'primary_genre']

Data Types:
show_id                  object
type                     object
title                    object
director                 object
cast                     object
country                  object
date_added       datetime64[ns]
release_year              int64
rating                   object
duration                 object
listed_in                object
description              object
year_added                int32
month_added              object
duration_int            float64
primary_genre            object
dtype: object

Missing Values:
show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added       0
release_year     0
rating           

In [28]:
print("Starting Data Cleaning...\n")

df['director'].fillna('Unknown', inplace=True)
df['cast'].fillna('Unknown', inplace=True)
df['country'].fillna('Unknown', inplace=True)
df['rating'].fillna(df['rating'].mode()[0], inplace=True)
df['duration'].fillna('Unknown', inplace=True)
df.dropna(subset=['date_added'], inplace=True)

if df['date_added'].dtype == object:
    df['date_added'] = df['date_added'].str.strip()
    df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
else:
    df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month_name()

df['duration_int'] = df['duration'].str.extract(r'(\d+)').astype(float)

df['primary_genre'] = df['listed_in'].str.split(',').str[0].str.strip()

df.dropna(subset=['date_added'], inplace=True)

print(f"Cleaning done! Remaining records: {df.shape[0]}")
print(f"\nMissing values after cleaning:")
print(df.isnull().sum())

Starting Data Cleaning...

Cleaning done! Remaining records: 8797

Missing values after cleaning:
show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added       0
release_year     0
rating           0
duration         0
listed_in        0
description      0
year_added       0
month_added      0
duration_int     3
primary_genre    0
dtype: int64


In [15]:
total = len(df)
movies = len(df[df['type'] == 'Movie'])
shows = len(df[df['type'] == 'TV Show'])
countries = df['country'].nunique()
genres = df['primary_genre'].nunique()
years_span = f"{int(df['year_added'].min())} - {int(df['year_added'].max())}"

print("=" * 50)
print("KEY PERFORMANCE INDICATORS (KPIs)")
print("=" * 50)
print(f"Total Titles      : {total}")
print(f"Total Movies      : {movies}")
print(f"Total TV Shows    : {shows}")
print(f"Countries Covered : {countries}")
print(f"Unique Genres     : {genres}")
print(f"Years on Netflix  : {years_span}")

KEY PERFORMANCE INDICATORS (KPIs)
Total Titles      : 8797
Total Movies      : 6131
Total TV Shows    : 2666
Countries Covered : 749
Unique Genres     : 36
Years on Netflix  : 2008 - 2021


In [14]:
type_counts = df['type'].value_counts().reset_index()
type_counts.columns = ['Type', 'Count']

fig1 = px.pie(type_counts, names='Type', values='Count',
              title='Movies vs TV Shows on Netflix',
              color_discrete_sequence=['#E50914', '#221F1F'],
              hole=0.4)
fig1.update_layout(template='plotly_dark')
fig1.show()

In [16]:
top_countries = df[df['country'] != 'Unknown']['country'].value_counts().head(10).reset_index()
top_countries.columns = ['Country', 'Count']

fig2 = px.bar(top_countries, x='Count', y='Country', orientation='h',
              title='Top 10 Countries by Netflix Content',
              color='Count', color_continuous_scale='Reds',
              text='Count')
fig2.update_layout(template='plotly_dark', yaxis={'categoryorder': 'total ascending'})
fig2.show()

In [13]:
yearly = df.groupby(['year_added', 'type']).size().reset_index(name='Count')

fig3 = px.line(yearly, x='year_added', y='Count', color='type',
               title='Netflix Content Added Over the Years',
               markers=True,
               color_discrete_sequence=['#E50914', '#B81D24'])
fig3.update_layout(template='plotly_dark')
fig3.show()

In [17]:
top_genres = df['primary_genre'].value_counts().head(10).reset_index()
top_genres.columns = ['Genre', 'Count']

fig4 = px.bar(top_genres, x='Genre', y='Count',
              title='Top 10 Content Genres on Netflix',
              color='Count', color_continuous_scale='OrRd',
              text='Count')
fig4.update_layout(template='plotly_dark', xaxis_tickangle=-30)
fig4.show()

In [18]:
rating_counts = df['rating'].value_counts().reset_index()
rating_counts.columns = ['Rating', 'Count']

fig5 = px.bar(rating_counts, x='Rating', y='Count',
              title='Content Rating Distribution',
              color='Count', color_continuous_scale='Reds',
              text='Count')
fig5.update_layout(template='plotly_dark')
fig5.show()

In [20]:
from plotly.subplots import make_subplots
import plotly.io as pio
with open("netflix_dashboard.html", "w") as f:
    f.write("<h1 style='text-align:center; font-family:Arial; color:#E50914'> Netflix AI Dashboard</h1>")
    f.write(fig1.to_html(full_html=False))
    f.write(fig2.to_html(full_html=False))
    f.write(fig3.to_html(full_html=False))
    f.write(fig4.to_html(full_html=False))
    f.write(fig5.to_html(full_html=False))

from google.colab import files
files.download('netflix_dashboard.html')
print("Dashboard exported and downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Dashboard exported and downloaded!
